In [37]:
# BOG(bag of words) == unique words
# TF = term frequecny
# IDF = inverse document frequency

About the Dataset:

1. id: unique id for a news article
2. title: the title of a news article
3. author: author of the news article
4. text: the text of the article; could be incomplete
5. label: a label that marks whether the news article is real or fake:
           1: Fake news
           0: real News

In [38]:
!which python

/home/midori/Desktop/Machine-Learning/venv/bin/python


In [39]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [40]:
import numpy as np
import pandas as pd

import nltk
import re
# regular expression

from nltk.corpus import stopwords
# Imports a predefined list of common words (like the, is, and, in, of) that usually carry little semantic meaning in NLP tasks.

from nltk.stem.porter import PorterStemmer
# Imports the Porter stemming algorithm.
# PorterStemmer → shrink words to their base form
# stemmer = PorterStemmer()
# stemmer.stem("running")   # -> 'run'    run, running, runs → same root
# stemmer.stem("studies")   # -> 'study'

from sklearn.feature_extraction.text import TfidfVectorizer

In [41]:
nltk.download('stopwords',download_dir='/home/midori/Desktop/Machine-Learning/')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/midori/Desktop/Machine-Learning/...
[nltk_data]   Package stopwords is already up-to-date!


True

In [42]:
nltk.data.path.append('/home/midori/Desktop/Machine-Learning/')

In [43]:
STOPWORDS = set(stopwords.words('english'))

In [44]:
# printing the stopwords in English
print(STOPWORDS)

{'itself', 'why', 'both', 'that', 'all', "mustn't", 'an', 'didn', 'under', 'hers', "we're", 'own', 'for', 'her', 'about', 'its', 'then', 'more', 'of', "it'll", 'll', 'which', 'ain', 'from', 'mightn', "shouldn't", 'any', 'can', "he's", 'ours', 'only', 'on', 'me', 'theirs', 'while', 'than', 'she', 'when', "wasn't", 'his', "aren't", 'being', "we've", 'few', "she'd", 'just', 'myself', 'have', 'how', 'your', 'isn', 'don', "he'll", 'this', 'he', 'is', 'some', 'between', 're', 's', "they'll", 'no', 'are', 'am', "he'd", 'him', 'up', "you'll", 'we', 'they', "doesn't", 'couldn', "you'd", 't', "you've", "we'll", 'other', 'wasn', 'once', 'there', "don't", "i'm", "they'd", 'o', 'again', 'will', "couldn't", 'a', 'been', 'such', 'himself', 'nor', 'here', 'hasn', 'm', 'into', "shan't", 've', 'yours', "that'll", 'themselves', 'won', 'same', "i'd", 'mustn', 'i', 'doesn', 'very', 'so', "needn't", 'down', 'now', "i'll", 'out', 'these', 'should', 'those', 'during', "hasn't", 'd', 'off', 'and', 'were', 'y',

### Loading Dataset

In [79]:
news_dataset = pd.read_csv('../../Data/news.csv')

In [80]:
news_dataset.head()

,id,title,author,text,label
0,0,House Dem Aide: We Didn’t Even See Comey’s Let...,Darrell Lucus,House Dem Aide: We Didn’t Even See Comey’s Let...,1
1,1,"FLYNN: Hillary Clinton, Big Woman on Campus - ...",Daniel J. Flynn,Ever get the feeling your life circles the rou...,0
2,2,Why the Truth Might Get You Fired,Consortiumnews.com,"Why the Truth Might Get You Fired October 29, ...",1
3,3,15 Civilians Killed In Single US Airstrike Hav...,Jessica Purkiss,Videos 15 Civilians Killed In Single US Airstr...,1
4,4,Iranian woman jailed for fictional unpublished...,Howard Portnoy,Print \nAn Iranian woman has been sentenced to...,1


### Data Preprocessing

In [81]:
news_dataset.isnull().sum()

id           0
title      558
author    1957
text        39
label        0
dtype: int64

In [82]:
# replacing the null values with empty string
news_dataset = news_dataset.fillna('')

In [83]:
news_dataset.isnull().sum()

id        0
title     0
author    0
text      0
label     0
dtype: int64

In [88]:
news_dataset['label'].value_counts()

label
1    10413
0    10387
Name: count, dtype: int64

In [91]:
news_dataset.groupby('label').size()

label
0    10387
1    10413
dtype: int64

In [50]:
news_dataset.shape

(20800, 5)

In [51]:
# merging the author name and news title
news_dataset['content'] = news_dataset['title']+' '+news_dataset['text']

In [52]:
news_dataset['content']

0        House Dem Aide: We Didn’t Even See Comey’s Let...
1        FLYNN: Hillary Clinton, Big Woman on Campus - ...
2        Why the Truth Might Get You Fired Why the Trut...
3        15 Civilians Killed In Single US Airstrike Hav...
4        Iranian woman jailed for fictional unpublished...
                               ...                        
20795    Rapper T.I.: Trump a ’Poster Child For White S...
20796    N.F.L. Playoffs: Schedule, Matchups and Odds -...
20797    Macy’s Is Said to Receive Takeover Approach by...
20798    NATO, Russia To Hold Parallel Exercises In Bal...
20799    What Keeps the F-35 Alive   David Swanson is a...
Name: content, Length: 20800, dtype: object

#### Stemming:

Stemming is the process of reducing a word to its Root word

example:
actor, actress, acting --> act

In [53]:
# port_stem.stem(word)
# running → run
# studies → studi
# played → play


#### PorterStemmer does **stemming**, not proper linguistic lemmatization.

So:

* `studies → studi` ❌ (not a real word)
* It just strips suffix patterns mechanically.

If you want:

* `studies → study` ✔

You need **lemmatization**, not stemming.

Example (using WordNetLemmatizer):

```python
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()
lemmatizer.lemmatize("studies", pos="v")
```

Stemming = crude rule-based chopping.
Lemmatization = dictionary + grammar aware reduction.

That’s the difference.


In [54]:
port_stem = PorterStemmer()

In [ ]:
def stemming(content):
    stemmed_content = re.sub('[^a-zA-Z]',' ',content)    # 1. keep letters only, replace others with space
    stemmed_content = stemmed_content.lower()            # 2. lowercase
    stemmed_content = stemmed_content.split()            # 3. split into tokens by whitespace
    # 4. remove stopwords and stem
    stemmed_content = [port_stem.stem(word) for word in stemmed_content if not word in STOPWORDS]  

    # 5. join back to string
    stemmed_content = ' '.join(stemmed_content) # stemmed_content = stemmed_content.apply(lambda x: ' '.join(x))  
    # .apply() is a Pandas Series/DataFrame method.
    
          
    return stemmed_content

In [56]:
news_dataset['content'] = news_dataset['content'].apply(stemming)


#### 1️⃣ What `re.sub()` Is Doing

```python
stemmed_content = re.sub('[^a-zA-Z]', ' ', content)
```

This does **cleaning**.

It removes:

* Numbers
* Punctuation
* Symbols
* Emojis
* Special characters


#### 2️⃣ What `PorterStemmer()` Is Doing

This line is the real stemming:

```python
port_stem.stem(word)
```

Example:

```
running → run
studies → studi
played → play
```

That is root reduction.

---

#### So Your Function Has 4 Separate Steps

1. **Regex cleaning** → remove non-letters
2. **Lowercasing** → standardize
3. **Stopword removal** → remove "the", "is", etc.
4. **Stemming** → reduce words to base form

#### Let’s Walk Through One Example

Input:

```python
"Running!!! quickly in the 2026 Olympics."
```

After regex:

```python
"Running     quickly in the     Olympics "
```

After lower + split:

```python
['running', 'quickly', 'in', 'the', 'olympics']
```

After stopword removal:

```python
['running', 'quickly', 'olympics']
```

After stemming:

```python
['run', 'quickli', 'olymp']
```

Final output:

```python
"run quickli olymp"
```

### Splitting X and Y

In [57]:
X = news_dataset['content']
X

0        hous dem aid even see comey letter jason chaff...
1        flynn hillari clinton big woman campu breitbar...
2        truth might get fire truth might get fire octo...
3        civilian kill singl us airstrik identifi video...
4        iranian woman jail fiction unpublish stori wom...
                               ...                        
20795    rapper trump poster child white supremaci rapp...
20796    n f l playoff schedul matchup odd new york tim...
20797    maci said receiv takeov approach hudson bay ne...
20798    nato russia hold parallel exercis balkan nato ...
20799    keep f aliv david swanson author activist jour...
Name: content, Length: 20800, dtype: object

In [58]:
Y = news_dataset['label']
Y

0        1
1        0
2        1
3        1
4        1
        ..
20795    0
20796    0
20797    0
20798    1
20799    1
Name: label, Length: 20800, dtype: int64

### TF-IDF_Vectorizer

In [74]:
# convert the textual data to Feature Vectors
vectorizer = TfidfVectorizer()

In [75]:
vectorizer.fit(X)
X = vectorizer.transform(X)

In [77]:
print(X)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 5115523 stored elements and shape (20800, 110766)>
  Coords	Values
  (0, 323)	0.044179022507208135
  (0, 518)	0.02332405077764721
  (0, 632)	0.039316203504174516
  (0, 869)	0.015976541921521512
  (0, 918)	0.016642095819834255
  (0, 1292)	0.021398757998419504
  (0, 1602)	0.018255643322868005
  (0, 1877)	0.11720929349150563
  (0, 3008)	0.04733009098564867
  (0, 3028)	0.019311970062028613
  (0, 3363)	0.011979048895968012
  (0, 3725)	0.03134889384323141
  (0, 4135)	0.016933166314925904
  (0, 4237)	0.019716700911550464
  (0, 4282)	0.02693212295173725
  (0, 4563)	0.017964895769890684
  (0, 4746)	0.026333626062969923
  (0, 4796)	0.042317596939954696
  (0, 4813)	0.01462055924601113
  (0, 6387)	0.018502204351742796
  (0, 6930)	0.0209635900182605
  (0, 8509)	0.021972639591348223
  (0, 9022)	0.014453700828335692
  (0, 10780)	0.048308058449603494
  (0, 12171)	0.02820114316501808
  :	:
  (20799, 106136)	0.03677728368716433
  (20799, 1062